# Go2 Backflip — Step-by-Step Tutorial

This notebook walks through running a **trained backflip policy** on the Unitree **Go2** quadruped in Genesis, section by section. By the end you will have:

1. Initialized Genesis on the **AMD ROCm** backend.
2. Built a single-environment Go2 scene with a phase-aware observation wrapper.
3. Loaded a TorchScript policy checkpoint (`single.pt` / `double.pt`).
4. Rolled the policy out and recorded an MP4.
5. Played the result back **inline** in this notebook.

The code mirrors `examples/locomotion/go2_backflip.py` but breaks the procedural script into teachable cells, each with explanatory prose.

> **Tip**: Run the cells from top to bottom. Each cell builds on the previous one's state.

## Prerequisites

- Run inside the **`genesis-world-amd:latest`** (auplc) container — see `examples/locomotion/README.md` for the one-line `docker run` that launches JupyterLab with `/dev/kfd` + `/dev/dri` already passed through.
- The notebook **must be opened from `examples/locomotion/`** (the JupyterLab quick-start command sets that as the working directory) because we import the `Go2Env` base class from `go2_env.py` and load policy checkpoints from `./backflip/`.
- Pre-trained checkpoints (`single.pt`, `double.pt`) need to live under `./backflip/` — see `backflip/readme.md` for the download link.

In [1]:
import os
import sys
from pathlib import Path

import torch

# Ensure ``go2_env`` is importable when the notebook is launched from a
# different working directory.
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import genesis as gs
from go2_env import Go2Env

print("torch:", torch.__version__)
print("genesis:", gs.__version__)
print("cwd:", NOTEBOOK_DIR)

[I 05/15/26 08:30:21.345 612] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


torch: 2.9.1+rocm7.11.0
genesis: 0.4.6
cwd: /opt/workspace/genesis-world/examples/locomotion


## 1 · Initialize Genesis (ROCm backend)

`gs.init(...)` selects a backend and sets up the JIT compiler. We pick `gs.amdgpu` to run on AMD GPUs via ROCm/HIP. The first run will JIT-compile a few kernels, which adds ~5–10 s of one-time startup; subsequent runs are cached.

In [2]:
gs.init(backend=gs.amdgpu)
print("Genesis device:", gs.device)

[Genesis] [08:30:23] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [08:30:23] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [08:30:23] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [08:30:23] [INFO] Running on [Radeon 8060S Graphics] with backend gs.amdgpu. Device memory: 96.00 GB.
[Genesis] [08:30:23] [INFO] 🚀 Genesis initialized. 🔖 version: 0.4.6, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
Genesis device: cuda:0


## 2 · Robot, observation and command configurations

Genesis locomotion environments are configured by four dicts:

| Dict | Purpose |
|------|---------|
| `env_cfg` | Robot-side parameters: number of actuated DoFs (`num_actions=12`), joint name list, **PD gains** (`kp=70`, `kd=3`), default joint angles (the prone "tuck" pose), termination thresholds, base init pose, episode length, etc. |
| `obs_cfg` | Observation vector size (`num_obs=60`) and per-channel scaling. |
| `reward_cfg` | Reward scales — empty here because we only **evaluate** a pre-trained policy; rewards aren't needed at inference time. |
| `command_cfg` | Velocity-command ranges for `Go2Env`. The backflip policy ignores commands but the base class still requires the dict. |

The 60-dim observation is built later in `BackflipEnv.get_observations`; we'll see how the layout matches `num_obs=60` exactly.

In [3]:
def get_cfgs():
    env_cfg = {
        "num_actions": 12,
        "default_joint_angles": {
            "FL_hip_joint": 0.0,
            "FR_hip_joint": 0.0,
            "RL_hip_joint": 0.0,
            "RR_hip_joint": 0.0,
            "FL_thigh_joint": 0.8,
            "FR_thigh_joint": 0.8,
            "RL_thigh_joint": 1.0,
            "RR_thigh_joint": 1.0,
            "FL_calf_joint": -1.5,
            "FR_calf_joint": -1.5,
            "RL_calf_joint": -1.5,
            "RR_calf_joint": -1.5,
        },
        "joint_names": [
            "FR_hip_joint", "FR_thigh_joint", "FR_calf_joint",
            "FL_hip_joint", "FL_thigh_joint", "FL_calf_joint",
            "RR_hip_joint", "RR_thigh_joint", "RR_calf_joint",
            "RL_hip_joint", "RL_thigh_joint", "RL_calf_joint",
        ],
        "kp": 70.0,
        "kd": 3.0,
        # Disable termination so the robot can flip past 90° pitch / 180° roll
        # without the env auto-resetting mid-flight.
        "termination_if_roll_greater_than": 1000,
        "termination_if_pitch_greater_than": 1000,
        "base_init_pos": [0.0, 0.0, 0.35],
        "base_init_quat": [0.0, 0.0, 0.0, 1.0],
        "episode_length_s": 20.0,        # overridden per-experiment below
        "resampling_time_s": 4.0,
        "action_scale": 0.5,
        "simulate_action_latency": True,
        "clip_actions": 100.0,
    }
    obs_cfg = {
        "num_obs": 60,
        "obs_scales": {
            "lin_vel": 2.0,
            "ang_vel": 0.25,
            "dof_pos": 1.0,
            "dof_vel": 0.05,
        },
    }
    reward_cfg = {"reward_scales": {}}
    command_cfg = {
        "num_commands": 3,
        "lin_vel_x_range": [0, 0],
        "lin_vel_y_range": [0, 0],
        "ang_vel_range": [0, 0],
    }
    return env_cfg, obs_cfg, reward_cfg, command_cfg


env_cfg, obs_cfg, reward_cfg, command_cfg = get_cfgs()
print("num_actions:", env_cfg["num_actions"])
print("num_obs    :", obs_cfg["num_obs"])

num_actions: 12
num_obs    : 60


## 3 · Pick the experiment

Two checkpoints are provided:

- **`single`** — one backflip, episode length **2 s**.
- **`double`** — two backflips back-to-back, episode length **3 s**.

We also assemble `camera_kwargs` for an offscreen 1280×720 camera. It must be created **before** `scene.build()` (Genesis bakes the renderer at build-time), so we set it up here and pass it to the env.

In [4]:
exp_name = "single"   # try "double" for the dual-flip policy

if exp_name == "single":
    env_cfg["episode_length_s"] = 2
elif exp_name == "double":
    env_cfg["episode_length_s"] = 3
else:
    raise ValueError(f"Unknown exp_name={exp_name!r}; expected 'single' or 'double'.")

camera_kwargs = dict(
    res=(1280, 720),
    pos=(2.5, 1.5, 1.2),
    lookat=(0.0, 0.0, 0.3),
    fov=40,
    GUI=False,           # offscreen camera; no native window
)

videos_dir = (NOTEBOOK_DIR / ".." / "videos").resolve()
videos_dir.mkdir(exist_ok=True)
output_video = str(videos_dir / f"go2_backflip_tutorial_{exp_name}.mp4")
print("Will save video to:", output_video)

Will save video to: /opt/workspace/genesis-world/examples/videos/go2_backflip_tutorial_single.mp4


## 4 · `BackflipEnv` — phase-aware observations

The vanilla `Go2Env.get_observations()` returns:

```text
[ang_vel(3) · grav(3) · cmd(3) · dof_pos(12) · dof_vel(12) · last_action(12) · last_last_action(12)]  → 57 dims
```

For backflips we drop the velocity command (the policy doesn't take user commands) and replace it with a **phase signal**: six sinusoids at periods T, 2T, and 4T, where T is the full episode length. This gives the network a clock so it can time the takeoff and landing precisely.

So the observation becomes:

```text
[ang_vel(3) · grav(3) · dof_pos(12) · dof_vel(12) · action(12) · last_action(12) · sin/cos × 3 freqs(6)]  → 60 dims
```

Subclassing keeps the rest of `Go2Env` (URDF loading, PD control, resets, stepping) intact.

In [5]:
class BackflipEnv(Go2Env):
    def get_observations(self):
        # Normalised time within the episode; multiply by π so it covers half a sine wave per episode.
        phase = torch.pi * self.episode_length_buf[:, None] / self.max_episode_length
        self.obs_buf = torch.cat(
            [
                self.base_ang_vel * self.obs_scales["ang_vel"],                        # 3
                self.projected_gravity,                                                # 3
                (self.dof_pos - self.default_dof_pos) * self.obs_scales["dof_pos"],    # 12
                self.dof_vel * self.obs_scales["dof_vel"],                             # 12
                self.actions,                                                          # 12
                self.last_actions,                                                     # 12
                torch.sin(phase), torch.cos(phase),                                    # 2  (T)
                torch.sin(phase / 2), torch.cos(phase / 2),                            # 2  (2T)
                torch.sin(phase / 4), torch.cos(phase / 4),                            # 2  (4T)
            ],
            axis=-1,
        )
        return self.obs_buf

    def step(self, actions):
        super().step(actions)
        self.get_observations()
        return self.obs_buf, self.rew_buf, self.reset_buf, self.extras

## 5 · Build the scene

Constructing `BackflipEnv` will:

1. Build a `gs.Scene` with simulation step `dt=0.02 s` (50 Hz control), 2 physics substeps, and self-collision **disabled** (it isn't needed for the airborne flip and saves compute).
2. Add the ground plane URDF and Go2 URDF, then add the offscreen camera.
3. Run `scene.build(n_envs=1)` — this is where Genesis JIT-compiles all the rigid-body kernels for AMDGPU. **First run can take 30 – 60 s** while shaders compile.
4. Configure the PD controllers with `kp=70`, `kd=3` on the 12 leg DoFs.
5. Call `cam.follow_entity(robot, fix_orientation=False)` so the recorded camera tracks the robot's body during the flip.

In [ ]:
env = BackflipEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    show_viewer=False,        # notebook is headless; rendering goes through the offscreen camera
    camera_kwargs=camera_kwargs,
)
env.cam.follow_entity(env.robot, fix_orientation=False)
print(f"Episode length: {env.max_episode_length} steps  ({env_cfg['episode_length_s']} s @ {1/env.dt:.0f} Hz)")

[Genesis] [08:30:25] [INFO] Scene <b11fbd0> created.
[Genesis] [08:30:25] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 0, uid: <f773dbc>, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.12/dist-packages/genesis/assets/urdf/plane/plane.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [08:30:25] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 1, uid: <6024051>, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.12/dist-packages/genesis/assets/urdf/go2/urdf/go2.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [08:30:26] [INFO] Building scene <b11fbd0>...
[Genesis] [08:30:30] [WARNING] Neutral robot position (qpos0) exceeds joint limits.


## 6 · Load the trained policy

The checkpoints under `./backflip/` are TorchScript artefacts (a frozen MLP). `torch.jit.load` deserialises the graph and weights; we then move it to Genesis's GPU device so we don't need to ferry observations across PCIe each step.

If you don't yet have the `.pt` files, see `backflip/readme.md` for the Drive folder.

In [ ]:
policy_path = NOTEBOOK_DIR / "backflip" / f"{exp_name}.pt"
assert policy_path.exists(), (
    f"Policy not found at {policy_path}. "
    "Download the .pt files into ./backflip/ — see backflip/readme.md."
)
policy = torch.jit.load(str(policy_path))
policy.to(device=gs.device)
print(f"Loaded policy: {policy_path.name}")

## 7 · Roll out the policy and record the video

The closed loop each step is:

1. `policy(obs)` → 12-D action.
2. `env.step(action)` advances the physics by 2 substeps (1 control tick) and returns the next observation.
3. `env.cam.render()` writes one frame into the camera's recording buffer.

We run for **two full episodes** (≈4 s for `single`, ≈6 s for `double`) so the video shows the robot landing, reset, and flipping again. `torch.no_grad()` skips autograd bookkeeping since we're only doing inference.

In [ ]:
fps = int(round(1.0 / env.dt))
total_steps = 2 * env.max_episode_length

env.cam.start_recording()
obs = env.reset()
try:
    with torch.no_grad():
        for step_i in range(total_steps):
            actions = policy(obs)
            obs, rews, dones, infos = env.step(actions)
            env.cam.render()
            if (step_i + 1) % env.max_episode_length == 0:
                print(f"  finished episode {(step_i + 1) // env.max_episode_length} / 2")
finally:
    env.cam.stop_recording(save_to_filename=output_video, fps=fps)
    print(f"Saved video → {output_video}")

## 8 · Watch the result inline

We use `IPython.display.Video` with `embed=True` so the MP4 bytes are base64-embedded into the notebook output — the clip will play even if you later open this notebook on a different machine where the file path no longer exists.

If the embedded video makes the notebook too large for git, switch to `embed=False` (just a `<video src="...">` reference to the file).

In [ ]:
from IPython.display import Video

Video(output_video, embed=True, width=720)

## 9 · Where to go next

- Re-run the notebook with `exp_name = "double"` (cell 3) to record the double-flip checkpoint.
- Set `show_viewer=True` in the env constructor (cell 5) when running locally with an X display to watch the rollout in real time alongside the recording.
- Crank up `num_envs` in cell 5 to evaluate many parallel rollouts at once — drop the camera (or render only env 0 via `vis_options.rendered_envs_idx=[0]`) for full speed.
- Train your own backflip from scratch with the reference repo: <https://github.com/ziyanx02/Genesis-backflip>.